# 京剧剧本解析 Notebook

这个 Notebook 现在是 **自包含** 的京剧剧本处理脚本。

它直接内嵌了解析管线，不再依赖外部 `opera_parsing_pipeline.py` 导入。

输入目录：
- 抽取后的剧本 JSON：`../data/dataSet`

输出目录：
- 聚合结果：`../public/json/opera`
- 分剧本结果：`../public/json/opera/*.json`
- 场次文本：`../public/chapters/opera`
- 原始正文文本：`../public/scripts/opera`

默认行为：
- 处理 **全部剧本文件**
- 默认不调用 LLM，先稳定跑通全量数据
- 如配置 `DEEPSEEK_API_KEY`，可切换 `USE_LLM=True`


## 一、内嵌解析管线

下面把京剧解析逻辑拆成多个代码格，方便逐段阅读、运行和调试。


In [ ]:
# 导入与基础依赖
# 这里放 Notebook 运行需要的标准库，以及可选的 DeepSeek / Pydantic 依赖。
import json
import math
import os
import re
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean
from typing import Any, Iterable

try:
    from langchain_openai import ChatOpenAI
    from pydantic import BaseModel, Field
except Exception:
    ChatOpenAI = None
    BaseModel = object
    Field = None


In [ ]:
# 文本模式与规则常量
# 这里集中管理场次正则、行当映射、情绪词表和颜色方案。
CHINESE_NUMERALS = "一二三四五六七八九十百千零"
SCENE_HEADER_RE = re.compile(r"^【([^】]+)】$")
SPEAKER_RE = re.compile(r"^([\u4e00-\u9fffA-Za-z0-9·〇○—－\-]{1,12})\s*[（(]")
ROLE_SPLIT_RE = re.compile(r"[:：]")
STAGE_DIRECTION_RE = re.compile(r"（([^）]+)）")

POSITIVE_WORDS = {
    "喜", "乐", "笑", "欢", "庆", "得意", "从容", "安", "胜", "吉", "忠", "义", "爱", "欢喜", "平安",
}
NEGATIVE_WORDS = {
    "怒", "恨", "悲", "哭", "惊", "愁", "怨", "怕", "死", "败", "失", "冤", "痛", "忧", "惨", "急", "惊疑",
}

ROLE_GROUP_MAP = {
    "老生": "生",
    "小生": "生",
    "武生": "生",
    "红生": "生",
    "娃娃生": "生",
    "青衣": "旦",
    "花旦": "旦",
    "刀马旦": "旦",
    "武旦": "旦",
    "老旦": "旦",
    "彩旦": "旦",
    "旦": "旦",
    "铜锤花脸": "净",
    "架子花脸": "净",
    "武花脸": "净",
    "净": "净",
    "文丑": "丑",
    "武丑": "丑",
    "丑": "丑",
}

GROUP_COLORS = {
    "生": ["#4E79A7", "#2F5597", "#6B8FD6"],
    "旦": ["#E15759", "#FF9DA7", "#C94C87"],
    "净": ["#59A14F", "#2E8B57", "#7AC36A"],
    "丑": ["#F28E2B", "#EDC948", "#D98C00"],
    "杂": ["#9C755F", "#BAB0AC", "#7F7F7F"],
}

DEFAULT_SAMPLE_KEYS = [
    "01001001_空城计.json",
    "01001003_三娘教子.json",
    "01005008_玉堂春.json",
    "01001002_洪羊洞.json",
    "01006007_定军山.json",
    "01002016_打渔杀家.json",
    "01005001_武家坡.json",
    "01002014_宇宙锋.json",
    "01003002_群英会.json",
    "01005006_六月雪.json",
    "01012007_贵妃醉酒.json",
    "70002105_赵氏孤儿.json",
    "11001001_连环套.json",
    "03031006_打面缸.json",
]


In [ ]:
# LLM 输出模型
# 只有在相关依赖可用时，才启用结构化输出的数据模型。
if ChatOpenAI is not None and Field is not None:
    class SceneAnalysis(BaseModel):
        summary: str = Field(description="一句话概括该场剧情")
        location: str = Field(description="本场主要地点，尽量使用剧中空间名称")
        importance: float = Field(description="本场重要性，0 到 1")
        conflict: float = Field(description="本场冲突强度，0 到 1")
        sentiment: float = Field(description="本场整体情绪，-1 到 1")


    class ChapterChunk(BaseModel):
        chapter: str = Field(description="章节名，如第1章")
        summary: str = Field(description="该章节的一句话概括")
        scene_numbers: list[int] = Field(description="纳入该章节的连续场次编号")
else:
    SceneAnalysis = None
    ChapterChunk = None


In [ ]:
# 基础数据与文件函数
# 这里定义创建 LLM、列出数据集文件、读取剧本 JSON 等基础函数。
def create_deepseek_llm(model: str = "deepseek-chat", temperature: float = 0.1):
    if ChatOpenAI is None:
        raise ImportError("langchain_openai 未安装，无法创建 DeepSeek LLM")
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise ValueError("未检测到 DEEPSEEK_API_KEY")
    return ChatOpenAI(
        model=model,
        temperature=temperature,
        api_key=api_key,
        base_url="https://api.deepseek.com",
    )


def list_dataset_files(dataset_root: str | Path) -> list[Path]:
    root = Path(dataset_root)
    return sorted(root.glob("*/*.json"))


def build_dataset_index(dataset_root: str | Path) -> dict[str, Path]:
    return {path.name: path for path in list_dataset_files(dataset_root)}


def load_play(path: str | Path) -> dict[str, Any]:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def safe_stem(name: str) -> str:
    stem = Path(name).stem
    stem = re.sub(r"[^\w\u4e00-\u9fff-]+", "_", stem)
    return stem.strip("_") or "opera"


def split_lines(text: str) -> list[str]:
    return [line.strip() for line in text.splitlines() if line.strip()]


In [ ]:
# 角色解析函数
# 这里把“主要角色”文本拆成角色名、行当和归一化分组。
def normalize_role_type(role_text: str) -> str:
    role_text = role_text.strip()
    for token in sorted(ROLE_GROUP_MAP.keys(), key=len, reverse=True):
        if token in role_text:
            return token
    return role_text.split()[0] if role_text else ""


def role_group(role_type: str) -> str:
    return ROLE_GROUP_MAP.get(role_type, "杂")


def parse_roles(raw_roles: str) -> list[dict[str, str]]:
    roles: list[dict[str, str]] = []
    for line in split_lines(raw_roles):
        parts = ROLE_SPLIT_RE.split(line, maxsplit=1)
        if len(parts) == 2:
            name, role = parts[0].strip(), parts[1].strip()
        else:
            name, role = line.strip(), ""
        role_type = normalize_role_type(role)
        roles.append(
            {
                "name": name,
                "role_type": role_type,
                "group": role_group(role_type),
                "raw": role,
            }
        )
    dedup = {}
    for role in roles:
        dedup[role["name"]] = role
    return list(dedup.values())


In [ ]:
# 场次切分与角色识别函数
# 这里负责按场次切分正文，并统计角色出现和引用。
def split_scenes(dialogue: str) -> list[dict[str, Any]]:
    lines = split_lines(dialogue)
    scenes: list[dict[str, Any]] = []
    current_title = "序场"
    current_lines: list[str] = []
    current_first_line = 1
    line_cursor = 1

    def flush_scene():
        nonlocal current_lines, current_title, current_first_line
        if not current_lines:
            return
        scenes.append(
            {
                "name": current_title,
                "text": "\n".join(current_lines).strip(),
                "firstLine": current_first_line,
                "lastLine": current_first_line + len(current_lines) - 1,
                "numLines": len(current_lines),
            }
        )
        current_lines = []

    for raw in lines:
        scene_match = SCENE_HEADER_RE.match(raw)
        if scene_match:
            flush_scene()
            current_title = scene_match.group(1).strip()
            current_first_line = line_cursor
            continue
        current_lines.append(raw)
        line_cursor += 1

    flush_scene()

    if not scenes:
        scenes.append(
            {
                "name": "全剧",
                "text": dialogue.strip(),
                "firstLine": 1,
                "lastLine": len(lines),
                "numLines": len(lines),
            }
        )

    for idx, scene in enumerate(scenes, start=1):
        scene["number"] = idx
    return scenes


def extract_speakers(scene_text: str) -> list[str]:
    speakers = []
    for line in split_lines(scene_text):
        match = SPEAKER_RE.match(line)
        if match:
            speakers.append(match.group(1).strip())
    return speakers


def count_character_mentions(scene_text: str, known_names: Iterable[str]) -> Counter:
    counter: Counter = Counter()
    for name in known_names:
        if not name:
            continue
        count = scene_text.count(name)
        if count:
            counter[name] += count
    for speaker in extract_speakers(scene_text):
        counter[speaker] += 2
    return counter


def extract_character_quote(scene_text: str, name: str) -> str:
    for line in split_lines(scene_text):
        if line.startswith(name):
            return line[:60]
    for line in split_lines(scene_text):
        if name in line:
            return line[:60]
    return ""


def sentiment_score(text: str) -> float:
    pos = sum(1 for word in POSITIVE_WORDS if word in text)
    neg = sum(1 for word in NEGATIVE_WORDS if word in text)
    total = pos + neg
    if total == 0:
        return 0.0
    return round((pos - neg) / total, 3)


def infer_emotion(text: str, score: float) -> str:
    if score >= 0.35:
        return "positive"
    if score <= -0.35:
        return "negative"
    if any(word in text for word in ["惊", "急", "怕", "危"]):
        return "tense"
    return "neutral"


In [ ]:
# 场次分析与评分函数
# 这里负责地点推断、场次摘要、情绪/冲突/重要性评估，以及可选 LLM 分析。
def heuristic_location(scene_name: str, scene_text: str) -> str:
    first_lines = split_lines(scene_text)[:6]
    for line in first_lines:
        for content in STAGE_DIRECTION_RE.findall(line):
            content = content.strip("。；，、 ")
            if len(content) <= 14 and any(token in content for token in ["宫", "城", "府", "营", "山", "亭", "堂", "殿", "江", "关", "坡", "寨", "楼", "店"]):
                return content
    place_matches = re.findall(r'([一-鿿]{1,8}(?:城|宫|府|营|山|亭|堂|殿|江|关|坡|寨|楼|店))', scene_text)
    if place_matches:
        return Counter(place_matches).most_common(1)[0][0]
    if any(token in scene_name for token in ["场", "出"]):
        return scene_name
    return scene_name or "未明地点"


def heuristic_scene_summary(scene_name: str, scene_text: str, characters: list[str]) -> str:
    chars = "、".join(characters[:3]) if characters else "角色"
    first_line = split_lines(scene_text)[:1]
    first_part = first_line[0][:28] if first_line else scene_name
    return f"{scene_name}：{chars}出场，围绕“{first_part}”展开。"


def heuristic_scene_metrics(scene_text: str, scene_name: str, characters: list[str]) -> dict[str, Any]:
    text = scene_name + "\n" + scene_text[:300]
    sentiment = sentiment_score(text)
    conflict_hits = sum(scene_text.count(token) for token in ["杀", "战", "怒", "恨", "冤", "逼", "退", "败", "哭", "惊"])
    importance = min(1.0, round(0.35 + math.log(len(scene_text) + 1, 10) / 3 + len(characters) * 0.03, 3))
    conflict = min(1.0, round(0.15 + min(conflict_hits, 12) / 15, 3))
    return {
        "summary": heuristic_scene_summary(scene_name, scene_text, characters),
        "location": heuristic_location(scene_name, scene_text),
        "importance": importance,
        "conflict": conflict,
        "sentiment": sentiment,
    }


def build_scene_prompt(play: dict[str, Any], scene: dict[str, Any], roles: list[dict[str, str]]) -> str:
    role_text = "\n".join(f"- {role['name']}：{role['role_type'] or '未知'}" for role in roles[:20])
    return f"""你是一位京剧叙事分析专家。请分析下面这一个场次，并仅返回 JSON。

【剧本】{play.get('剧本名字', '未知剧本')}
【情节】{play.get('情节', '')[:500]}
【主要角色】
{role_text}

【场次名称】{scene['name']}
【场次文本】
{scene['text'][:2200]}

输出字段：summary, location, importance, conflict, sentiment。
- summary：一句中文总结
- location：简洁地点，如“西城城楼”
- importance/conflict：0 到 1
- sentiment：-1 到 1
"""


def analyze_scene_with_llm(llm, play: dict[str, Any], scene: dict[str, Any], roles: list[dict[str, str]]) -> dict[str, Any]:
    if SceneAnalysis is None:
        return heuristic_scene_metrics(scene["text"], scene["name"], [])
    try:
        structured = llm.with_structured_output(SceneAnalysis)
        result = structured.invoke(build_scene_prompt(play, scene, roles))
        return {
            "summary": result.summary,
            "location": result.location,
            "importance": float(result.importance),
            "conflict": float(result.conflict),
            "sentiment": float(result.sentiment),
        }
    except Exception:
        return heuristic_scene_metrics(scene["text"], scene["name"], [])


def chunk_scene_numbers(scene_count: int) -> list[list[int]]:
    if scene_count <= 2:
        chapter_count = 1
    elif scene_count <= 5:
        chapter_count = 2
    elif scene_count <= 9:
        chapter_count = 3
    elif scene_count <= 16:
        chapter_count = 4
    else:
        chapter_count = 5
    chunk_size = math.ceil(scene_count / chapter_count)
    numbers = list(range(1, scene_count + 1))
    return [numbers[i:i + chunk_size] for i in range(0, scene_count, chunk_size)]


def group_scenes_with_llm(llm, scenes: list[dict[str, Any]]) -> list[dict[str, Any]]:
    if ChapterChunk is None:
        return []
    scene_listing = "\n".join(
        f"{scene['number']}. {scene['name']} — {scene['summary']}" for scene in scenes
    )
    prompt = f"""请把下面这些京剧场次按叙事阶段划分成 2 到 5 个连续章节，仅返回 JSON。

{scene_listing}

输出格式：
{{
  "chapters": [
    {{"chapter": "第1章", "summary": "章节概括", "scene_numbers": [1,2]}}
  ]
}}

要求：
- 每个 scene_numbers 必须连续
- 覆盖全部场次且不重复
- summary 要概括该阶段的叙事推进
"""
    try:
        class ChapterPlan(BaseModel):
            chapters: list[ChapterChunk] = Field(description="章节划分")
        structured = llm.with_structured_output(ChapterPlan)
        plan = structured.invoke(prompt)
        return [
            {
                "chapter": item.chapter,
                "summary": item.summary,
                "scene_numbers": item.scene_numbers,
            }
            for item in plan.chapters
        ]
    except Exception:
        return []


In [ ]:
# 章节、角色、地点聚合函数
# 这里把场次组织成章节，并构造角色、地点和链接结构。
def build_chapters(scenes: list[dict[str, Any]], use_llm: bool = False, llm=None) -> list[dict[str, Any]]:
    plan = group_scenes_with_llm(llm, scenes) if use_llm and llm is not None else []
    if not plan:
        plan = [
            {
                "chapter": f"第{i + 1}章",
                "summary": "；".join(scenes[n - 1]["name"] for n in chunk),
                "scene_numbers": chunk,
            }
            for i, chunk in enumerate(chunk_scene_numbers(len(scenes)))
        ]

    scene_by_number = {scene["number"]: scene for scene in scenes}
    chapters: list[dict[str, Any]] = []

    for chapter in plan:
        chapter_scenes = [scene_by_number[n] for n in chapter["scene_numbers"] if n in scene_by_number]
        if not chapter_scenes:
            continue
        char_counter: Counter = Counter()
        loc_counter: Counter = Counter()
        pair_counter: Counter = Counter()
        for scene in chapter_scenes:
            for character in scene["characters"]:
                char_counter[character["name"]] += 1
            loc_counter[scene["location"]] += 1
            names = [character["name"] for character in scene["characters"]]
            for i in range(len(names)):
                for j in range(i + 1, len(names)):
                    pair = tuple(sorted((names[i], names[j])))
                    pair_counter[pair] += 1
        chapters.append(
            {
                "chapter": chapter["chapter"],
                "numScenes": len(chapter_scenes),
                "numLines": sum(scene["numLines"] for scene in chapter_scenes),
                "summary": chapter["summary"],
                "conflict": round(mean(scene["ratings"]["conflict"] for scene in chapter_scenes), 3),
                "importance": round(mean(scene["ratings"]["importance"] for scene in chapter_scenes), 3),
                "locations": dict(loc_counter),
                "characters": dict(char_counter),
                "links": [
                    {
                        "source": source,
                        "target": target,
                        "value": value,
                        "interaction": f"{source}与{target}在该章节共同出场 {value} 次",
                    }
                    for (source, target), value in sorted(pair_counter.items(), key=lambda item: (-item[1], item[0]))
                ],
                "_scene_numbers": list(chapter["scene_numbers"]),
            }
        )
    return chapters


def stable_color(name: str, group: str) -> str:
    palette = GROUP_COLORS.get(group, GROUP_COLORS["杂"])
    index = sum(ord(char) for char in name) % len(palette)
    return palette[index]


def character_short_name(name: str) -> str:
    if len(name) <= 3:
        return name
    return name[:2]


def build_characters(role_items: list[dict[str, str]], scene_map: dict[str, list[dict[str, Any]]]) -> list[dict[str, Any]]:
    role_lookup = {item["name"]: item for item in role_items}
    all_names = set(role_lookup) | set(scene_map)
    characters = []
    for name in sorted(all_names, key=lambda value: (-len(scene_map.get(value, [])), value)):
        role_item = role_lookup.get(name, {"role_type": "", "group": "杂"})
        role_type = role_item.get("role_type", "")
        group = role_item.get("group", "杂")
        quotes = [scene_item["quote"] for scene_item in scene_map.get(name, []) if scene_item.get("quote")]
        quote = quotes[0] if quotes else f"{name} — {role_type or '未标注'}"
        characters.append(
            {
                "character": name,
                "short": character_short_name(name),
                "key": name,
                "quote": quote,
                "group": group,
                "color": stable_color(name, group),
                "explanation": [
                    f"行当: {role_type or '未标注'}",
                    f"所属: {group}组",
                    f"出场场次: {len(scene_map.get(name, []))}",
                ],
            }
        )
    return characters


def build_locations(scenes: list[dict[str, Any]]) -> list[dict[str, str]]:
    locations = []
    seen = set()
    for scene in scenes:
        name = scene["location"]
        if not name or name in seen:
            continue
        seen.add(name)
        locations.append({"name": name, "key": name, "quote": f"场景发生在{name}"})
    return locations


def process_play(
    path: str | Path,
    *,
    llm=None,
    use_llm: bool = False,
    type_label: str = "京剧剧本",
    default_year: int = 1900,
) -> dict[str, Any]:
    play = load_play(path)
    roles = parse_roles(play.get("主要角色", ""))
    role_lookup = {item["name"]: item for item in roles}

    raw_scenes = split_scenes(play.get("正文对话", ""))
    scenes: list[dict[str, Any]] = []
    scene_map: dict[str, list[dict[str, Any]]] = defaultdict(list)

    known_names = list(role_lookup.keys())

    for raw_scene in raw_scenes:
        mention_counter = count_character_mentions(raw_scene["text"], known_names)
        present_names = [name for name, count in mention_counter.most_common() if count > 0]
        if not present_names:
            present_names = extract_speakers(raw_scene["text"])[:6]

        analysis = (
            analyze_scene_with_llm(llm, play, raw_scene, roles)
            if use_llm and llm is not None
            else heuristic_scene_metrics(raw_scene["text"], raw_scene["name"], present_names)
        )

        max_count = max(mention_counter.values()) if mention_counter else 1
        scene_characters = []
        for rank, name in enumerate(present_names, start=1):
            count = mention_counter.get(name, 1)
            importance = round(count / max_count, 3) if max_count else 0.2
            quote = extract_character_quote(raw_scene["text"], name)
            rating = sentiment_score(quote or raw_scene["text"][:80])
            role_type = role_lookup.get(name, {}).get("role_type", "")
            item = {
                "name": name,
                "importance": importance,
                "importance_rank": rank,
                "emotion": infer_emotion(quote or raw_scene["text"], rating),
                "quote": quote,
                "rating": rating,
                "role": role_type,
            }
            scene_characters.append(item)
            scene_map[name].append(item)

        scenes.append(
            {
                "number": raw_scene["number"],
                "name": raw_scene["name"],
                "location": analysis["location"],
                "characters": scene_characters,
                "summary": analysis["summary"],
                "firstLine": raw_scene["firstLine"],
                "lastLine": raw_scene["lastLine"],
                "numLines": raw_scene["numLines"],
                "chapter": "",
                "ratings": {
                    "importance": round(float(analysis["importance"]), 3),
                    "conflict": round(float(analysis["conflict"]), 3),
                    "sentiment": round(float(analysis["sentiment"]), 3),
                },
                "text": raw_scene["text"],
            }
        )

    chapters = build_chapters(scenes, use_llm=use_llm, llm=llm)
    scene_to_chapter = {}
    for chapter in chapters:
        for scene_number in chapter.get("_scene_numbers", []):
            scene_to_chapter[scene_number] = chapter["chapter"]
        chapter.pop("_scene_numbers", None)
    for scene in scenes:
        scene["chapter"] = scene_to_chapter.get(scene["number"], "第1章")
        scene.pop("text", None)

    characters = build_characters(roles, scene_map)
    locations = build_locations(scenes)

    raw_title = play.get("剧本名字") or Path(path).stem
    title = raw_title.split("（一名：", 1)[0]
    result = {
        "title": title,
        "type": type_label,
        "author": play.get("source_folder_name", "未知来源"),
        "year": default_year,
        "url": "",
        "image": "",
        "num_chapters": len(chapters),
        "num_scenes": len(scenes),
        "num_characters": len(characters),
        "num_locations": len(locations),
        "source_file": Path(path).name,
        "source_folder": play.get("source_folder", ""),
        "plot_summary": play.get("情节", ""),
        "performance_notes": play.get("注释", ""),
        "chapters": chapters,
        "scenes": scenes,
        "characters": characters,
        "locations": locations,
    }
    return result


In [ ]:
# 单剧本处理与导出函数
# 这里负责处理单个剧本，并把结果写到 public 目录。
def export_story_texts(result: dict[str, Any], raw_play: dict[str, Any], public_root: str | Path) -> None:
    root = Path(public_root)
    stem = safe_stem(result["source_file"])
    script_dir = root / "scripts" / "opera"
    chapter_dir = root / "chapters" / "opera" / stem
    script_dir.mkdir(parents=True, exist_ok=True)
    chapter_dir.mkdir(parents=True, exist_ok=True)

    (script_dir / f"{stem}.txt").write_text(raw_play.get("正文对话", ""), encoding="utf-8")

    raw_scenes = split_scenes(raw_play.get("正文对话", ""))
    for scene in raw_scenes:
        scene_name = re.sub(r"[^\w\u4e00-\u9fff-]+", "_", scene["name"]).strip("_") or f"scene_{scene['number']}"
        file_name = f"scene_{scene['number']:02d}_{scene_name}.txt"
        (chapter_dir / file_name).write_text(scene["text"], encoding="utf-8")


def write_outputs(
    results: dict[str, dict[str, Any]],
    *,
    dataset_root: str | Path,
    public_root: str | Path,
    src_data_root: str | Path | None = None,
    bundle_name: str = "opera-samples.json",
) -> dict[str, str]:
    public_root = Path(public_root)
    dataset_index = build_dataset_index(dataset_root)
    json_dir = public_root / "json" / "opera"
    json_dir.mkdir(parents=True, exist_ok=True)

    for key, result in results.items():
        (json_dir / key).write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        raw_play = load_play(dataset_index[key])
        export_story_texts(result, raw_play, public_root)

    public_bundle = json_dir / bundle_name
    public_bundle.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

    written = {"public_bundle": str(public_bundle)}
    if src_data_root is not None:
        src_bundle = Path(src_data_root) / bundle_name
        src_bundle.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
        written["src_bundle"] = str(src_bundle)
    return written


In [ ]:
# 批量处理入口函数
# 这里提供全量遍历所有剧本文件的入口。
def load_default_selection(dataset_root: str | Path) -> list[Path]:
    index = build_dataset_index(dataset_root)
    return [index[key] for key in DEFAULT_SAMPLE_KEYS if key in index]


def process_many(
    paths: Iterable[str | Path],
    *,
    llm=None,
    use_llm: bool = False,
) -> dict[str, dict[str, Any]]:
    results = {}
    for path in paths:
        result = process_play(path, llm=llm, use_llm=use_llm)
        results[Path(path).name] = result
    return results


## 二、运行配置

这里设置输入目录、输出目录，以及是否启用 DeepSeek。


In [ ]:
from pathlib import Path
import json
import os

# 赛题数据输入目录：extract_opera.py 抽取后的全部剧本 JSON
DATASET_ROOT = Path("../data/dataSet").resolve()

# 输出目录：只写到 public，不再同步其他目录
PUBLIC_ROOT = Path("../public").resolve()

# 是否启用 DeepSeek：默认关闭，先保证全量剧本稳定跑通
USE_LLM = False

# 聚合结果文件名
BUNDLE_NAME = "opera-dataset.json"

assert DATASET_ROOT.exists(), DATASET_ROOT
assert PUBLIC_ROOT.exists(), PUBLIC_ROOT


In [ ]:
# 全量读取全部剧本文件
all_files = list_dataset_files(DATASET_ROOT)
selected_paths = all_files

print(f"dataset files: {len(all_files)}")
print(f"selected files: {len(selected_paths)}")
print([path.name for path in selected_paths[:10]])


In [ ]:
# 根据配置决定是否启用 DeepSeek
llm = None
if USE_LLM:
    if not os.getenv("DEEPSEEK_API_KEY"):
        secrets_path = Path("../secrets.json")
        if secrets_path.exists():
            secrets = json.loads(secrets_path.read_text(encoding="utf-8"))
            if secrets.get("deepseek"):
                os.environ["DEEPSEEK_API_KEY"] = secrets["deepseek"]
    llm = create_deepseek_llm()
    print("DeepSeek enabled.")
else:
    print("Running in heuristic mode (no LLM calls).")


## 三、执行处理

先处理全部剧本，再写出到 `public`。


In [ ]:
# 处理全部剧本
results = process_many(selected_paths, llm=llm, use_llm=USE_LLM)

print(f"processed: {len(results)}")
first_key = next(iter(results))
print("first key:", first_key)
print(json.dumps({
    "title": results[first_key]["title"],
    "num_chapters": results[first_key]["num_chapters"],
    "num_scenes": results[first_key]["num_scenes"],
    "num_characters": results[first_key]["num_characters"],
    "num_locations": results[first_key]["num_locations"],
}, ensure_ascii=False, indent=2))


In [ ]:
# 将结果写到 public 目录
written = write_outputs(
    results,
    dataset_root=DATASET_ROOT,
    public_root=PUBLIC_ROOT,
    bundle_name=BUNDLE_NAME,
)
written


## 四、结果抽样检查

抽查第一部剧本的章节和场次结构是否合理。


In [ ]:
sample_key = next(iter(results))
sample = results[sample_key]

print("sample:", sample_key)
print("title:", sample["title"])
print("chapters:", sample["num_chapters"], "scenes:", sample["num_scenes"])
print("first chapter:")
print(json.dumps(sample["chapters"][0], ensure_ascii=False, indent=2))
print("first scene:")
print(json.dumps(sample["scenes"][0], ensure_ascii=False, indent=2))


In [ ]:
# 说明：
# 1. 当前 Notebook 默认处理全部剧本文件。
# 2. 当前 Notebook 默认只输出到 ../public。
# 3. 如需启用 DeepSeek，请把 USE_LLM 改为 True，并配置 DEEPSEEK_API_KEY。
